In [1]:
import os
from dotenv import load_dotenv
from reanimator.core import Reanimator
from reanimator.labelers import OpenAILabeler, LocalModelLabeler, TopicChunkPair, calculate_cohens_kappa
from reanimator.retrieval import Indexer, Retriever, reciprocal_rank_fusion, run_experiment
from reanimator.models import save_judgements, load_judgements, Document

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
import pyterrier as pt
import nltk

load_dotenv()

import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /Users/mr_kk/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
!python3 -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 12.3 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [3]:
! pip install -U spacy


[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [4]:
import os 
os.environ["OPENAI_API_KEY"] = "dummy_key"  # Replace with your actual OpenAI API key or set it in your environment variables

In [5]:
# Initialize Reanimator with dummy parameters (we won't use the download functionality)
reanimator = Reanimator(
    irds_name="irds:cord19/trec-covid",  # This won't be used
    email="dummy@gmail.com",
    config={
        "downloader": {
            "email": "dummy@gmail.com"
        }
    }
)

INFO: OpenAILabeler initialized with model: gpt-4.1-mini-2025-04-14


Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/Users/mr_kk/reanim_test/Reanimator/src/reanimator/sources.py:18: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [ ]:
# Skip the dataset loading and create a document for your ten.pdf
pdf_path = "data/pdfs/test.pdf"  # Your PDF file path

# Create a document object manually
doc = Document(
    doc_id="test",  # Custom document ID
    pdf_path=pdf_path  # Path to your PDF file
)

# Create a list with just this one document
docs = [doc]

# Set accelerator options
accelerator_options = AcceleratorOptions(
    num_threads=8, device=AcceleratorDevice.MPS
)

print(f"Using PDF file: {pdf_path}")

Using PDF file: data/pdfs/ten.pdf


In [7]:
# Extract content from the PDF (this will include formulas)
print(f"Extracting content from {pdf_path}...")
reanimator.extract_content(docs, accelerator_options)

# Check if formulas were extracted
if docs[0].formulas:
    print(f"\n Success! Found {len(docs[0].formulas)} formulas:")
    print("=" * 60)

else:
    print(" No formulas found in the PDF.")
    print("This could mean:")
    print("1. The PDF doesn't contain mathematical formulas")
    print("2. The formulas are images rather than text")
    print("3. There might be an issue with the extraction")

# Save the extracted document
reanimator.save_documents(docs, "documents_new1")
print(f"\nDocument saved to 'documents' folder")

Extracting content from data/pdfs/ten.pdf...

Step 3: Extracting content from PDFs...


Extracting Content:   0%|          | 0/1 [00:00<?, ?it/s]2025-10-04 13:59:03,983 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-04 13:59:04,011 - INFO - Going to convert document batch...
2025-10-04 13:59:04,012 - INFO - Initializing pipeline for StandardPdfPipeline with options hash b482b7b5f70e963efe771f8708610e1b
2025-10-04 13:59:04,015 - INFO - Loading plugin 'docling_defaults'
2025-10-04 13:59:04,016 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-10-04 13:59:04,020 - INFO - Loading plugin 'docling_defaults'
2025-10-04 13:59:04,025 - INFO - Registered ocr engines: ['easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-10-04 13:59:04,173 - INFO - Accelerator device: 'mps'
2025-10-04 13:59:06,004 - INFO - Accelerator device: 'mps'
2025-10-04 13:59:06,975 - INFO - Accelerator device: 'mps'
2025-10-04 13:59:07,336 - INFO - Removing MPS from available devices because it is not in supported_devices=[<AcceleratorDevice.CPU: 'cpu'>, <AcceleratorDevic


 Success! Found 10 formulas:
Saving 1 documents to directory documents_new1...
Finished saving documents.

Document saved to 'documents' folder
